In [2]:
import pickle

# Leaderboard user_ids from each country with >500pp
with open("leaderboard_ids.pickle", "rb") as handle:
    user_ids = pickle.load(handle)

user_ids = set(user_ids)

# User ids that have already been scraped. NOT real time. Needs to be reset to empty set upon rerunning this nb.
with open("completed_user_ids.pickle", "rb") as handle:
    completed_user_ids = pickle.load(handle)
completed_user_ids = set(completed_user_ids)    

    
remaining_user_ids = user_ids - completed_user_ids

In [89]:
import boto3

dynamodb = boto3.client("dynamodb")

user_id = 28956125

response = dynamodb.query(
    TableName="osu_scores",
    KeyConditionExpression="user_id = :uid",
    ExpressionAttributeValues={
        ":uid": {"N": str(user_id)}
    }
)

items = response["Items"]


# Delete all items with a specific user_id (testing purposes)
def delete_items_with_user_id(user_id):
    # Scan the table to retrieve all items with the specified user_id
    response = dynamodb.scan(
        TableName="osu_scores",
        FilterExpression="user_id = :uid",
        ExpressionAttributeValues={
            ":uid": {"N": str(user_id)}
        }
    )

    # Iterate over the retrieved items and delete each one
    for item in response["Items"]:
        dynamodb.delete_item(
            TableName="osu_scores",
            Key={
                "user_id": {"N": str(user_id)},
                "beatmap_id": item["beatmap_id"]
            }
        )

    # Check if there are more items to scan
    while "LastEvaluatedKey" in response:
        response = dynamodb.scan(
            TableName="osu_scores",
            FilterExpression="user_id = :uid",
            ExpressionAttributeValues={
                ":uid": {"N": str(user_id)}
            },
            ExclusiveStartKey=response["LastEvaluatedKey"]
        )

        # Iterate over the retrieved items and delete each one
        for item in response["Items"]:
            dynamodb.delete_item(
                TableName="osu_scores",
                Key={
                    "user_id": {"N": str(user_id)},
                    "beatmap_id": item["beatmap_id"]
                }            )

user_id_to_delete = user_id
delete_items_with_user_id(user_id_to_delete)

items 

[]

In [100]:
def transform_to_dynamodb_format(item):
    """
    Transform a dictionary item to DynamoDB format.
    """
    dynamodb_item = {}
    for key, value in item.items():
        if value is None:
            dynamodb_item[key] = {"NULL": True}
        elif isinstance(value, str):
            dynamodb_item[key] = {"S": value}
        elif isinstance(value, int):
            dynamodb_item[key] = {"N": str(value)}
        elif isinstance(value, float):
            dynamodb_item[key] = {"N": str(value)}
        else:
            raise TypeError(f"Unsupported type for key {key}: {type(value)}")
    return dynamodb_item

def batch_insert_items(items):
    request_items = {
        "osu_scores": [
            {
                "PutRequest": {
                    "Item": transform_to_dynamodb_format(item)
                }
            }
            for item in items
        ]
    }
    response = dynamodb.batch_write_item(RequestItems=request_items)
    # Check if there are unprocessed items
    unprocessed_items = response.get("UnprocessedItems", {})
    while unprocessed_items:
        response = dynamodb.batch_write_item(RequestItems=unprocessed_items)
        unprocessed_items = response.get("UnprocessedItems", {})


# batch_insert_items(items_to_insert)

In [106]:
import os
from ossapi import Ossapi
from numpy import array_split
from time import strftime, localtime
import time
from datetime import datetime

CLIENT_ID = os.environ.get("OSU_CLIENT_ID")
CLIENT_SECRET = os.environ.get("OSU_CLIENT_SECRET")

ossapi = Ossapi(CLIENT_ID, CLIENT_SECRET)

user_ids = list(remaining_user_ids)
num_partitions = 1
partitioned_user_ids = array_split(user_ids, num_partitions)

num_done = 0
last_time = time.time()


def scrape_users(ids):
    """
    Adds user top scores to dynamodb.
    """
    global num_done
    global last_time
    epoch_time = datetime(1970, 1, 1)

    for user_id in ids:
        print(user_id)

        try:
            top_scores = ossapi.user_scores(user_id, type="best", mode="osu", limit=100)
            scores = []

            for score in top_scores:
                try:
                    beatmap = getattr(score, "beatmap", None)
                    beatmap_id = getattr(beatmap, "id", None)
                    beatmapset = getattr(score, "beatmapset", None)
                    beatmapset_id = getattr(beatmapset, "id", None)
                    mods = getattr(score, "mods", None)
                    mods_value = getattr(mods, "value", None)
                    pp = getattr(score, "pp", None)
                    created_at = getattr(score, "created_at", None)
                    created_at_str = (
                        datetime.strftime(created_at, "%Y-%m-%d %H:%M:%S")
                        if created_at
                        else None
                    )
                    mode = getattr(score, "mode_int", None)
                    item = {
                        "user_id": int(user_id),
                        "beatmapset_id": beatmapset_id,
                        "beatmap_id": beatmap_id,
                        "pp": pp,
                        "created_at": created_at_str,
                        "mods": mods_value,
                        "mode": mode,
                    }
                    scores.append(item)

                    if len(scores) == 25:
                        try:
                            batch_insert_items(scores)
                            scores = []
                        except Exception as e:
                            print(f"Error inserting scores: {e}")
                            continue

                except Exception as e:
                    print(e)
                    print(f"Error creating score object: {e}")
                    continue

        except Exception as e:
            print(e)
            continue

        num_done += 1
        if num_done % 5 == 0:
            print(
                str(num_done) + ": " + str(time.time() - last_time),
                strftime("%H:%M:%S", localtime(time.time())),
            )
            last_time = time.time()

In [107]:
user_ids = partitioned_user_ids[0][:100]
scrape_users(user_ids)

9437184
30408706
19922948
30408712
24117256
5: 57.1677930355072 23:24:54
12582925


KeyboardInterrupt: 